# 00d — Assemble independent finding annotations

Use the graph environment; no GPU or foundation model is required. Requires `candidate_preparation.py` beside this notebook.

Match existing Experiment 4 ECG machine interpretations by **patient and study ID**, using enriched Symile partitions or the original ECG source manifest. Merge them with 00b CXR annotations. Generated PULSE/MedGemma reports are never read as reference targets.

The pilot ECG parser uses the vocabulary from Experiment 4. These are noisy machine/report-derived targets, not expert ground truth. Missing, uncertain, historical, conflicting, and unmentioned findings remain unknown. Coverage is saved per case; no patients are dropped. If no usable ECG reference concepts can be matched, this notebook stops without creating annotations.json. The parser and any mapping changes need development-set clinical audit before testing.

In [ ]:
from pathlib import Path
import json, sys
NOTEBOOK_DIR=Path.cwd().resolve()
PROJECT_ROOT=next((p for p in (NOTEBOOK_DIR,*NOTEBOOK_DIR.parents) if (p/'config.json').is_file()),None)
if PROJECT_ROOT is None:raise FileNotFoundError('Start in graph_reasoning or its extraction subfolder; project config.json is required')
HELPER_ROOT=next((p for p in (NOTEBOOK_DIR,PROJECT_ROOT,PROJECT_ROOT/'symile_extraction_update') if (p/'candidate_preparation.py').is_file() and (p/'sensor_candidate_generation.py').is_file()),None)
if HELPER_ROOT is None:raise FileNotFoundError('Copy both preparation helpers into the project or extraction subfolder')
sys.path.insert(0,str(HELPER_ROOT))
from candidate_preparation import *
CONFIG=read(PROJECT_ROOT/'config.json')
def resolve(value):
    p=Path(value).expanduser()
    return (p if p.is_absolute() else PROJECT_ROOT/p).resolve()
SOURCES=CONFIG.get('export_sources',{})
MANIFEST=resolve(SOURCES.get('manifest','data/prepared/manifest.json'))
SENSORS=MANIFEST.parent/'sensor_sources.json'
ARTIFACT_ROOT=resolve(CONFIG.get('embedding_artifact_root','/path/to/data/symile/ecg_cxr_bridge_artifacts'))
GENERATION_DIR=resolve(CONFIG.get('candidate_generation_dir','outputs/symile_sensor_only_v1'))
assert MANIFEST.is_file() and SENSORS.is_file(), 'Run notebook 00b first'


In [ ]:
ANNOTATIONS=resolve(SOURCES.get('annotations','data/prepared/annotations.json'))
CXR_ANNOTATIONS=MANIFEST.parent/'cxr_annotations.json'
print('Independent ECG source availability:')
for s in ('train','val','test'):
    p=ARTIFACT_ROOT/'experiment4'/'partition'/'symile'/f'{s}.jsonl'
    print(p, p.is_file())
p=ARTIFACT_ROOT/'external'/'ecg'/'manifest.jsonl'
print(p,p.is_file())
# Optional direct native MIMIC ECG metadata if Experiment 4 lacks matching reference text.
# Set to your real machine_measurements.csv or .csv.gz path; never a generated-report file.
ECG_MEASUREMENTS_CSV=CONFIG.get('ecg_reference_csv')
summary=build_annotations(MANIFEST,SENSORS,CXR_ANNOTATIONS,ARTIFACT_ROOT,ANNOTATIONS,ecg_measurements_csv=ECG_MEASUREMENTS_CSV)
print(summary)
print('Saved:',ANNOTATIONS)
print('Review annotation_coverage.json before using labels for evaluation.')
